In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

In [ ]:

@dataclass
class Llama4TextConfig:
    vocab_size: int = 202048
    hidden_size: int = 5120
    intermediate_size: int = 8192
    intermediate_size_mlp: int = 16384
    num_hidden_layers: int = 48
    num_attention_heads: int = 40
    num_key_value_heads: int = 8
    head_dim: int = 128
    max_position_embeddings: int = 4096 * 32
    rms_norm_eps: float = 1e-5
    pad_token_id: int = 200018
    bos_token_id: int = 1
    eos_token_id: int = 2
    rope_theta: float = 500000
    attention_dropout: float = 0.0
    num_experts_per_tok: int = 1
    num_local_experts: int = 16
    use_qk_norm: bool = True
    no_rope_layer_interval: int = 4
    attention_chunk_size: int = 8192
    attn_temperature_tuning: float = 4
    floor_scale: int = 8192
    attn_scale: float = 0.1

@dataclass
class Llama4VisionConfig:
    hidden_size: int = 768
    num_hidden_layers: int = 34
    num_attention_heads: int = 16
    num_channels: int = 3
    intermediate_size: int = 5632
    vision_output_dim: int = 7680
    image_size: int = 448
    patch_size: int = 14
    norm_eps: float = 1e-5
    pixel_shuffle_ratio: float = 0.5
    projector_input_dim: int = 4096
    projector_output_dim: int = 4096
    projector_dropout: float = 0.0
    attention_dropout: float = 0.0
    rope_theta: int = 10000


In [ ]:
class Llama4TextExperts(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.num_experts = config.num_local_experts
        self.intermediate_size = config.intermediate_size
        self.hidden_size = config.hidden_size
        self.expert_dim = config.intermediate_size
        self.gate_up_proj = nn.Parameter(torch.empty(self.num_experts, self.hidden_size, 2 * self.expert_dim))
        self.down_proj = nn.Parameter(torch.empty(self.num_experts, self.expert_dim, self.hidden_size))
        self.act_fn = nn.SiLU()

        nn.init.normal_(self.gate_up_proj)
        nn.init.normal_(self.down_proj)

    def forward(self, hidden_states):

        hidden_states = hidden_states.reshape(self.num_experts, -1, self.hidden_size)

        gate_up = torch.bmm(hidden_states, self.gate_up_proj)

        gate, up = gate_up.chunk(2, dim=-1)

        gated = up * self.act_fn(gate)

        next_states = torch.bmm(gated, self.down_proj)

        next_states = next_states.view(-1, self.hidden_size)

        return next_states

In [ ]:
class Llama4TextMLP(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.config = config
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
        self.activation_fn = nn.SiLU()

    def forward(self, x):
        gated = self.activation_fn(self.gate_proj(x)) * self.up_proj(x)
        return self.down_proj(gated)

In [ ]:
class Llama4TextMoe(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.top_k = config.num_experts_per_tok
        self.hidden_dim = config.hidden_size
        self.num_experts = config.num_local_experts

        self.experts = Llama4TextExperts(config)
        self.router = nn.Linear(config.hidden_size, config.num_local_experts, bias=False)
        self.shared_expert = Llama4TextMLP(config)



    def forward(self, hidden_states):

        batch_size, seq_len, embed_dim = hidden_states.shape

        hidden_states = hidden_states.view(-1, self.hidden_dim)             # batch_size*seq_len, embed_dim

        router_logits = self.router(hidden_states)                          # batch_size*seq_len, num_local_experts

        # step 1: pretend each expert will get all tokens
        tokens_per_expert = batch_size * seq_len

        # step 2: get the topk experts for each token
        router_top_value, router_indices = torch.topk(router_logits, self.top_k, dim=-1)        # batch_size*seq_len, num_experts_per_tok

        # step 3: create a matrix of -infinities
        router_scores = torch.full_like(router_logits, float("-inf")).scatter_(dim=1, index=router_indices, src=router_top_value).transpose(0,1)        # (batch_size*seq_len, num_local_experts).T

        # step 4: because we are passing in "ALL TOKENS" to every expert, lets update our router indicies to be
        # indexes from 0 to NUM TOKENS, repeated for EVERY Expert! it will something like:

        # [0, 1, 2, ..., Num Tokens]
        # [0, 1, 2, ..., Num Tokens]
        # [0, 1, 2, ..., Num Tokens]
        # ...

        router_indicies = torch.arange(tokens_per_expert, device=hidden_states.device).unsqueeze(0).expand(router_scores.size(0), -1)           # num_local_experts, batch_size*seq_len

        # step 5: grab embeddings with these indexes, but we have an embed dimension as well
        router_indicies = router_indicies.reshape(-1, 1).expand(-1, self.hidden_dim)        # num_local_experts*batch_size*seq_len, self.hidden_dim

        # scale 6: scale our router weights
        router_scores = torch.sigmoid(router_scores.float()).to(hidden_states.dtype)        # num_local_experts, batch_size*seq_len

        # scale 7: gather our hidden states by our router indices
        router_in = torch.gather(
            input=hidden_states,
            dim=0,
            index=router_indicies,
        )       # num_local_experts*batch_size*seq_len, self.hidden_dim


        router_in = router_in * router_scores.reshape(-1, 1)            # num_local_experts*batch_size*seq_len, self.hidden_dim

        # MOE
        router_out = self.experts(router_in)                            # num_local_experts*batch_size*seq_len, self.hidden_dim

        # shared expert
        shared_expert_out = self.shared_expert(hidden_states)           # batch_size*seq_len, self.hidden_dim

        shared_expert_out.scatter_add_(dim=0, index=router_indicies, src=router_out)

        return shared_expert_out


In [ ]:
class Llama4TextRotaryEmbedding(nn.Module):

    def __init__(self, config, device=None):
        super().__init__()

        self.config = config
        self.max_seq_len_cached = config.max_position_embeddings
        self.original_max_seq_len = config.max_position_embeddings

        inv_freq = self._compute_default_rope_parameters(self.config, device=device)
        self.register_buffer("inv_freq", inv_freq, persistent=False)


    def _compute_default_rope_parameters(self, config, device):

        base = config.rope_theta
        head_dim = config.head_dim

        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).to(device=device, dtype=torch.float) / head_dim))

        return inv_freq


    @torch.no_grad()
    def forward(self, x, position_ids):

        inv_freq_expanded = self.inv_freq[None, :, None].expand(position_ids.shape[0], -1, 1).to(x.device)       # (batch_size, head_dim/2, 1)

        position_ids_expanded = position_ids[:, None, :].float()                        #  (batch_size, 1, seq_len)

        with torch.autocast(device_type=x.device.type, enabled=False):
            freqs = (inv_freq_expanded @ position_ids_expanded)                         # (batch_size, head_dim/2, seq_len)
            freqs = freqs.transpose(1, 2)                                               # (batch_size, seq_len, head_dim/2)

            freqs_cis = torch.polar(abs=torch.ones_like(freqs), angle=freqs)            # (batch_size, seq_len, head_dim/2)

        return freqs_cis

# [theta1,
#  theta2,
#  theta3,
#  ...   ,
#  ...   ,
#  theta(d/2)]


# [0, 1, 2, 3, 4, 5]


# @ -> [0, theta1, 2theta1, 3theta1, 4theta1, 5theta1]
#      [0, theta2, 2theta2, 3theta2, 4theta2, 5theta2]
#      [0, theta3, 2theta3, 3theta3, 4theta3, 5theta3]
#      ...
#      ...
#      [0, theta(d/2), 2theta(d/2), 3theta(d/2), 4theta(d/2), 5theta(d/2)]


# .T -> [0,            0,        0, ...,           0]
#       [theta1,  theta2,   theta3, ...,  theta(d/2)]
#       [2theta1, 2theta2, 2theta3, ..., 2theta(d/2)]
#       [3theta1, 3theta2, 3theta3, ..., 3theta(d/2)]
#       [4theta1, 4theta2, 4theta3, ..., 4theta(d/2)]
#       [5theta1, 5theta2, 5theta3, ..., 5theta(d/2)]


# .polar() -> [cos(0)+jsin(0),                         cos(0)+jsin(0),             cos(0)+jsin(0), ...,                     cos(0)+jsin(0)]
#             [cos(theta1)+jsin(theta1),     cos(theta2)+jsin(theta2),   cos(theta3)+jsin(theta3), ...,   cos(theta(d/2))+jsin(theta(d/2))]
#             [cos(2theta1)+jsin(2theta1), cos(2theta2)+jsin(2theta2), cos(2theta3)+jsin(2theta3), ..., cos(2theta(d/2))+jsin(2theta(d/2))]
#             [cos(3theta1)+jsin(3theta1), cos(3theta2)+jsin(3theta2), cos(3theta3)+jsin(3theta3), ..., cos(3theta(d/2))+jsin(3theta(d/2))]
#             [cos(4theta1)+jsin(4theta1), cos(4theta2)+jsin(4theta2), cos(4theta3)+jsin(4theta3), ..., cos(4theta(d/2))+jsin(4theta(d/2))]
#             [cos(5theta1)+jsin(5theta1), cos(5theta2)+jsin(5theta2), cos(5theta3)+jsin(5theta3), ..., cos(5theta(d/2))+jsin(5theta(d/2))]
# = freqs_cis

In [ ]:
def apply_rotary_emb(xq, xk, freqs_cis):

    # xq -> (batch_size, seq_len, num_heads, head_dim)

    xq_split = xq.float().reshape(*xq.shape[:-1], -1, 2)
    xk_split = xk.float().reshape(*xk.shape[:-1], -1, 2)

    # xq_split -> (batch_size, seq_len, num_heads, head_dim/2, 2)

    xq_split = torch.view_as_complex(xq_split)
    xk_split = torch.view_as_complex(xk_split)

    # xq_split  -> (batch_size, seq_len, num_heads, head_dim/2)
    # freqs_cis -> (batch_size, seq_len,         1, head_dim/2)

    xq_out = xq_split * freqs_cis[:, :, None, :]
    xk_out = xk_split * freqs_cis[:, :, None, :]

    xq_out = torch.view_as_real(xq_out).flatten(3)
    xk_out = torch.view_as_real(xk_out).flatten(3)

    return xq_out.type_as(xq), xk_out.type_as(xk)

In [ ]:
class Llama4TextL2Norm(nn.Module):

    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def _norm(self, x):
        # we want to do x / sqrt((x**2).mean()) along the last dimension!
        return x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)

    def forward(self, x):
        return self._norm(x.float()).type_as(x)

In [ ]:
class Llama4TextRMSNorm(nn.Module):

    def __init__(self, hidden_size, eps=1e-5):
        super().__init__()

        self.eps = eps
        self.hidden_size = hidden_size
        self.weight = nn.Parameter(torch.ones(hidden_size))

    def forward(self, x):
        output = self._norm(x.float()).type_as(x)
        return output * self.weight

    def _norm(self, x):
        return x * torch.rsqrt(x.pos(2).mean(dim=-1, keepdim=True) + self.eps)

In [ ]:
class Cache:

    def __init__(self, config):
        super().__init__()
        self._seen_tokens = 0
        self.key_cache = [torch.tensor([]) for _ in range(config.num_hidden_layers)]
        self.value_cache = [torch.tensor([]) for _ in range(config.num_hidden_layers)]

    def __repr__(self):
        return f"DyanmicCache(Num_Layers: {len(self.key_cache)} | Cached Tokens: {self.key_cache[0].shape[2]})"

    def update(self, key_states, value_states, layer_idx):
        # (batch_size, num_heads, seq_len, head_dim)

        if layer_idx == 0:
            self._seen_tokens += 1

        self.key_cache[layer_idx] = torch.cat([self.key_cache[layer_idx], key_states], dim=-2)
        self.value_cache[layer_idx] = torch.cat([self.value_cache[layer_idx], value_states], dim=-2)

        return self.key_cache[layer_idx], self.value_cache[layer_idx]

    def get_seq_len(self, layer_idx=0):
        return self.key_cache.shape[layer_idx].shape[-2] if self.key_cache[layer_idx].numel() != 0 else 0


In [ ]:
class Llama4TextAttention(nn.Module):

    def __init__(self, config, layer_idx):
        super().__init__()

        self.config = config
        self.layer_idx = layer_idx
        self.head_dim = config.head_dim
        self.num_attention_heads = config.num_attention_heads   # 12, yani 12 tane query head var
        self.num_key_value_heads = config.num_key_value_heads   # 3, yani 3 tane key head, 3 tane value head var
        self.num_key_value_groups = config.num_attention_heads // config.num_key_value_heads    # 12//3=4, 4 tane q head tek bir kv head ile calisacak
        self.scaling = self.head_dim ** -0.5
        self.attn_scale = config.attn_scale
        self.floor_scale = config.floor_scale
        self.attn_temperature_tuning = config.attn_temperature_tuning
        self.attention_dropout = config.attention_dropout
        self.is_causal = True
        self.use_rope = int((layer_idx + 1) % 4 == 0)

        self.q_rpoj = nn.Linear(config.hidden_size, config.num_attention_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, config.num_key_value_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(config.num_attention_heads * self.head_dim, config.hidden_size, bias=False)

        if self.config.use_qk_norm and self.use_rope:
            self.qk_norm = Llama4TextL2Norm()
        else:
            self.qk_norm = None


    def _repeat_kv(self, hidden_states, n_rep):

        batch_size, num_heads, seq_len, embed_dim = hidden_states.shape

        hidden_states = hidden_states[:, :, None, :, :].expand(batch_size, num_heads, n_rep, seq_len, embed_dim)
        hidden_states = hidden_states.reshape(batch_size, num_heads*n_rep, seq_len, embed_dim)

        return hidden_states


    def forward(self,
                hidden_states,
                postion_embeddings=None,
                attention_mask=None,
                past_key_value=None,
                cache_position=None):

        # hidden_states -> batch_size, seq_len, hidden_size

        input_shape = hidden_states.shape[:-1]      # batch_size, seq_len

        hidden_shape = (*input_shape, -1, self.head_dim)        # batch_size, seq_len, -1, head_dim

        query_states = self.q_proj(hidden_states).reshape(hidden_shape)     # batch_size, seq_len, num_attention_heads * head_dim -> batch_size, seq_len, num_attention_heads, head_dim
        key_states = self.k_proj(hidden_states).reshape(hidden_shape)

        value_states = self.v_proj(hidden_states).reshape(hidden_shape).transpose(1, 2)

        if self.use_rope:
            query_states, key_states = apply_rotary_emb(query_states, key_states, postion_embeddings.to(query_states.device))

        if self.qk_norm is not None:
            query_states = self.qk_norm(query_states)
            key_states = self.qk_norm(key_states)

        cache_position = cache_position.unsqueeze(0).expand(hidden_states.shape[0], -1)

        if self.attn_temperature_tuning and not self.use_rope:
            attn_scales = (
                torch.log(torch.floor((cache_position.float() + 1) / self.floor_scale) + 1.0) * self.attn_scale + 1.0
            )
            attn_scales = attn_scales.view((*input_shape, 1, 1))
            query_states = (query_states * attn_scales).to(query_states.dtype)

        query_states = query_states.transpose(1, 2)     # batch_size, num_attention_heads, seq_len, head_dim
        key_states = key_states.transpose(1, 2)         # batch_size, num_key_value_heads, seq_len, head_dim

        if past_key_value is not None:
            key_states, value_states = past_key_value.update(key_states, value_states, self.layer_idx)

        # GQA
        # step 1: repeat keys/values heads to match the query heads
        # query: (batch_size, attention_heads, seq_len, head_dim)
        # value: (batch_size, num_key_value_heads, seq_len, head_dim)
        # value: (batch_size, num_key_value_heads, seq_len, head_dim)

        key_states = self._repeat_kv(key_states, self.num_key_value_groups)                     # batch_size, num_attention_heads, seq_len, head_dim
        value_states = self._repeat_kv(value_states, self.num_key_value_groups)

        attn_weights = torch.matmul(query_states, key_states.transpose(2, 3)) * self.scaling     # batch_size, num_attention_heads, seq_len, seq_len

        if attention_mask is not None:      # batch_size, 1, seq_len, seq_len
            causal_mask = attention_mask[:, :, :, :key_states.shape[-2]]

            attn_weights = attn_weights + causal_mask

        attn_weights = nn.functional.softmax(attn_weights.float(), dim=-1).to(query_states.dtype)

        attn_output = torch.matmul(attn_weights, value_states)              # batch_size, num_attention_heads, seq_len, head_dim

        attn_output = attn_output.transpose(1, 2).flatten(2)                # batch_size, seq_len, num_attention_heads, head_dim -> batch_size, seq_len, num_attention_heads * head_dim

        attn_output = self.o_proj(attn_output)                              # batch_size, seq_len, hidden_size

        return attn_output                                                  # batch_size, seq_len, hidden_size

In [ ]:
class Llama4TextDecoderLayer(nn.Module):

    def __init__(self, config, layer_idx):
        super().__init__()

        self.config = config
        self.layer_idx = layer_idx
        self.hidden_size = config.hidden_size
        self.self_attn = Llama4TextAttention(config, layer_idx)
        self.use_chunked_attention = int((layer_idx + 1) % 4 != 0)
        self.feed_forward = Llama4TextMoe(config)
        self.input_layernorm = Llama4TextRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = Llama4TextRMSNorm(config.hidden_size, eps=config.rms_norm_eps)


    def forward(self,
                 hidden_states,
                 attention_mask=None,
                 chunk_causal_mask=None,
                 past_key_value=None,
                 cache_position=None,
                 position_embeddings=None):

        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)

        if self.use_chunked_attention and chunk_causal_mask is not None:
            attention_mask = chunk_causal_mask

        attention_states = self.self_attn(hidden_states=hidden_states,
                                          position_embeddings=position_embeddings,
                                          attention_mask=attention_mask,
                                          past_key_value=past_key_value,
                                          cache_position=cache_position)

        hidden_states = residual + attention_states

        residual = hidden_states

        hidden_states = self.post_attention_layernorm(hidden_states)

        hidden_states = self.feed_forward(hidden_states)

        hidden_states = residual + hidden_states.view(residual.shape)

        return hidden_states

In [ ]:
class Llama4TextModel(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.config = config
        self.padding_idx = config.pad_token_id
        self.vocab_size = config.vocab_size

        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=self.padding_idx)

        self.layers = nn.ModuleList(
            [
                Llama4TextDecoderLayer(config, layer_idx)
                for layer_idx in range(config.num_hidden_layers)
            ]
        )

        self.norm = Llama4TextRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.rotary_emb = Llama4TextRotaryEmbedding(config)


    def forward(self,
                input_ids=None,
                input_embeds=None,
                attention_mask=None,
                position_ids=None,
                past_key_value=None,
                cache_position=None):

        if past_key_value is None:
            past_key_value = Cache(self.config)

        if input_embeds is None:
            hidden_states = self.embed_tokens(input_ids)

        else:
            hidden_states = input_embeds

        if cache_position is None:
            past_seen_tokens = past_key_value.get_seq_len()
            cache_position = torch.arange(past_seen_tokens, past_seen_tokens + hidden_states.shape[1], device=hidden_states.device)

        if position_ids is None:
            position_ids = cache_position.unsqueeze(0)

        # update our causal mask
        causal_mask, chunked_attention_mask = self._update_causal_mask(
            attention_mask, hidden_states, cache_position
        )

        freqs_cis = self.rotary_emb(hidden_states, position_ids)

        for layer in self.layers:
            hidden_states = layer(
                hidden_states,
                attention_mask,
                chunked_attention_mask if chunked_attention_mask is not None else causal_mask,
                cache_position,
                freqs_cis
            )

            hidden_states = self.norm(hidden_states)

            return hidden_states, past_key_value



    def _update_causal_mask(self,
                            attention_mask,
                            input_tensor,
                            cache_position):

        seq_len = input_tensor.shape[1]
        attention_chunk_size = self.config.attention_chunk_size


        start_cache_pos = cache_position[0]


        causal_mask = self._prepare_4d_causal_attention_mask_with_cache_position(
            attention_mask,
            sequence_length=seq_len,
            cache_position=cache_position,
            batch_size=input_tensor.shape[0],
            dtype=input_tensor.dtype,
            device=input_tensor.device
        )


        cond1 = start_cache_pos >= attention_chunk_size
        cond2 = (start_cache_pos < attention_chunk_size) & (start_cache_pos + seq_len > attention_chunk_size)


        key_length = torch.where(
            cond1,
            attention_chunk_size - 1 + seq_len,
            torch.where(cond2, start_cache_pos + seq_len, attention_chunk_size)
        )


        chunked_attention_mask = None
        if seq_len > self.config.attention_chunk_size:

            chunked_attention_mask = self._create_chunked_attention_mask(
                self.config.attention_chunk_size,
                start = start_cache_pos,
                end = start_cache_pos+key_length,
                device = input_tensor.device,
            )

            chunked_attention_mask = chunked_attention_mask & attention_mask

            chunked_attention_mask = (
                chunked_attention_mask[None, None, :, :]
                .to(input_tensor.dtype)
                .masked_fill(chunked_attention_mask, torch.finfo(input_tensor.dtype).min)
            )

        return causal_mask, chunked_attention_mask


    def _create_chunked_attention_mask(self,
                                       attention_chunk_size,
                                       start,
                                       end,
                                       device):

        block_pos = torch.abs(
            (torch.arange(start, end).unsqueeze(0)//attention_chunk_size) -
            (torch.arange(start, end).unsqueeze(1)//attention_chunk_size)
        )

        # [0, 1, 2, 3, 4, 5] -> [0, 0, 0, 1, 1, 1]
        # -
        # [0,    [0,
        #  1,     0,
        #  2,     0,
        #  3,     1,
        #  4,     1,
        #  5]  -> 1]
        #  =
        #  [ 0,  0,  0, 1, 1, 1]
        #  [ 0,  0,  0, 1, 1, 1]
        #  [ 0,  0,  0, 1, 1, 1]
        #  [-1, -1, -1, 0, 0, 0]
        #  [-1, -1, -1, 0, 0, 0]
        #  [-1, -1, -1, 0, 0, 0] -> block_pos

        # tok_pos ->  [ 0,  1,  2,  3,  4, 5]
        #             [-1,  0,  1,  2,  3, 4]
        #             [-2, -1,  0,  1,  2, 3]
        #             [-3, -2, -1,  0,  1, 2]
        #             [-4, -3, -2, -1,  0, 1]
        #             [-5, -4, -3, -2, -1, 0]

        # block_pos == 0
        # -> [True, True, True, False, False, False]
        #    [True, True, True, False, False, False]
        #    [True, True, True, False, False, False]
        #    [False, False, False, True, True, True]
        #    [False, False, False, True, True, True]
        #    [False, False, False, True, True, True]

        # tok_pos <= 0
        # -> [True, False, False, False, False, False]
        #    [True,  True, False, False, False, False]
        #    [True,  True,  True, False, False, False]
        #    [True,  True,  True,  True, False, False]
        #    [True,  True,  True,  True,  True, False]
        #    [True,  True,  True,  True,  True,  True]

        # & ->
        # [True,  False, False, False, False, False]
        # [True,   True, False, False, False, False]
        # [True,   True,  True, False, False, False]
        # [False, False, False,  True, False, False]
        # [False, False, False,  True,  True, False]
        # [False, False, False,  True,  True,  True] -> mask


        tok_pos = torch.arange(start, end).unsqueeze(0) - torch.arange(start, end).unsqueeze(1)

        mask = (block_pos == 0) & (tok_pos <= 0)

        return mask


    def _prepare_4d_causal_attention_mask_with_cache_position(
        self,
        attention_mask,
        sequence_length,
        cache_position,
        batch_size,
        dtype,
        device
    ):

        causal_mask = torch.full(
            (sequence_length, sequence_length), fill_value=torch.finfo(dtype).min, dtype=dtype, device=device
        )

        if sequence_length != 1:
            causal_mask = torch.triu(causal_mask, diagonal=1)

        causal_mask *= torch.arange(sequence_length, device=device) > cache_position.to(device).reshape(-1, 1)

        causal_mask = causal_mask[None, None, :, :].expand(batch_size, 1, -1, -1)

        if attention_mask is not None:
            causal_mask = causal_mask.clone()

            mask_length = attention_mask.shape[-1]

            padding_mask = causal_mask[:, :, :, :mask_length] + attention_mask[:, None, None, :].to(device)
            padding_mask = (padding_mask==0)
            causal_mask[:, :, :, :mask_length] = causal_mask[:, :, :, :mask_length].masked_fill(padding_mask, torch.finfo(dtype).min)

        return causal_mask

In [ ]:
class Llama4ForCausalLM(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.model = Llama4TextModel(config)
        self.vocab_size = config.vocab_size
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size)

    def forward(self,
                input_ids=None,
                input_embeds=None,
                attention_mask=None,
                position_ids=None,
                past_key_value=None):

        outputs, past_key_value = self.model(input_ids,
                                              input_embeds,
                                              attention_mask=attention_mask,
                                              position_ids=position_ids,
                                              past_key_value=past_key_value)

        logits = self.lm_head(outputs)

        return logits, past_key_value

In [ ]:
class Llama4VisionMLP(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.config = config
        self.activation_fn = nn.GELU()
        self.fc1 = nn.Linear(config.hidden_size, config.intermediate_size, bias=True)
        self.fc2 = nn.Linear(config.intermediate_size, config.hidden_size, bias=True)

    def forward(self, hidden_states):

        hidden_states = self.fc1(hidden_states)
        hidden_states = self.activation_fn(hidden_states)
        hidden_states = self.fc2(hidden_states)
        return hidden_states


In [ ]:
class Llama4VisionMLP2(nn.Module):

    def __init__(self, in_features, out_features, config):
        super().__iniit__()

        self.fc1 = nn.Linear(in_features, out_features, bias=False)
        self.fc2 = nn.Linear(out_features, out_features, bias=False)
        self.activation_fn = nn.GELU()
        self.dropout = nn.Dropout(config.projector_dropout)

    def forward(self, hidden_states):
        hidden_states = self.activation_fn(self.fc1(hidden_states))
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.activation_fn(self.fc2(hidden_states))
        return hidden_states

In [ ]:
class Llama4MultiModalProjection(nn.Module):

    def __init__(self, vision_config, text_config):
        super().__init__()

        self.linear_1 = nn.Linear(
            vision_config.projector_output_dim,
            text_config.hidden_size,
            bias=False
        )

    def forward(self, x):
        return self.linear_1(x)


In [ ]:
class Llama4VisionPixelShuffleMLP(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.pixel_shuffle_ratio = config.pixel_shuffle_ratio
        self.inner_dim = int(config.hidden_size // (self.pixel_shuffle_ratio**2))
        self.output_dim = config.projector_output_dim

        self.mlp = Llama4VisionMLP2(self.inner_dim, self.output_dim, config)


    def _pixel_shuffle(self, x):

        batch_size, num_patches, embed_dim = x.shape

        patch_size = int(num_patches ** 0.5)

        x = x.reshape(batch_size, patch_size, patch_size, embed_dim)

        batch_size, height, width, channels = x.shape

        x = x.reshape(batch_size, height, int(width * self.pixel_shuffle_ratio), int(channels / self.pixel_shuffle_ratio))

        x = x.transpose(1, 2)

        x = x.reshape(batch_size, int(width*self.pixel_shuffle_ratio), int(height*self.pixel_shuffle_ratio), int(channels/self.pixel_shuffle_ratio**2))

        x = x.permute(0, 2, 1, 3)

        x = x.reshape(batch_size, -1, x.shape[-1])

        return x


    def forward(self, x):

        shuffled = self._pixel_shuffle(x)

        proj = self.mlp(shuffled)

        return proj



In [ ]:
class Llama4VisionRotaryEmbedding(nn.Module):

    def __init__(self, config):
        super().__init__()

        index = config.image_size // config.patch_size      # index = 4 // 2 = 2
        image_index = torch.arange(index**2, dtype=torch.int32).reshape(index**2, 1)

        # [0,
        #  1,
        #  2,
        #  3] -> image_index

        image_index = torch.cat([image_index, torch.tensor([-2]).reshape(-1, 1)], dim=0)

        # [0,
        #  1,
        #  2,
        #  3
        # -2] -> image_index

        x_coord = image_index % index
        y_coord = image_index // index

        # [0,
        #  1,
        #  0,
        #  1,
        #  0] -> x_coord

        # [0,
        #  0,
        #  1,
        #  1,
        #  -1] -> y_coord

        freq_dim = config.hidden_size // config.num_attention_heads // 2

        # hidden_size = 16
        # num_attention_heads = 2

        # -> head_dim = 8
        # -> [a, b, c, d, e, f, g, h]

        # -> [a, b]
        #    [c, d]
        #    [e, f]
        #    [g, h]

        # -> freq_dim = 4


        rope_freq = 1.0 / (config.rope_theta ** (torch.arange(0, freq_dim, 2)[:(freq_dim//2)].float() / freq_dim))

        # [0, 2][:2] -> [0, 0.5] -> [1, 100] -> [1, 0.01] -> rope_freq

        freqs_x = ((x_coord + 1)[..., None] * rope_freq[None, None, :]).repeat_interleave(2, dim=-1)
        freqs_y = ((y_coord + 1)[..., None] * rope_freq[None, None, :]).repeat_interleave(2, dim=-1)

        # --- freqs_x ---
        # [1,
        #  2,
        #  1,
        #  2,
        #  1] -> freqs_x, shape= 5, 1, 1

        # [1, 0.01] -> rope_freq, shape 1, 1, 2

        # * -> [[1, 0.01],
        #       [2, 0.02],
        #       [1, 0.01],
        #       [2, 0.02],
        #       [1, 0.01]] -> freqs_x, shape = 5, 1, 2

        # .repeat_interleave() -> [[1, 1, 0.01, 0.01],
        #                          [2, 2, 0.02, 0.02],
        #                          [1, 1, 0.01, 0.01],
        #                          [2, 2, 0.02, 0.02],
        #                          [1, 1, 0.01, 0.01]] -> freqs_x, shape = 5, 1, 4


        # --- freqs_y ---

        # [1,
        #  1,
        #  2,
        #  2,
        #  0] -> freqs_y, shape= 5, 1, 1

        # [1, 0.01] -> rope_freq, shape 1, 1, 2

        # [[1, 0.01],
        #  [1, 0.01],
        #  [2, 0.02],
        #  [2, 0.02],
        #  [0,  0.0]] -> freqs_y, shape = 5, 1, 2

        #  .repeat_interleave() -> [[1, 1, 0.01, 0.01],
        #                           [1, 1, 0.01, 0.01],
        #                           [2, 2, 0.02, 0.02],
        #                           [2, 2, 0.02, 0.02],
        #                           [0, 0,  0.0,  0.0]] -> freqs_y, shape = 5, 1, 4



        freqs = torch.cat([freqs_x, freqs_y], dim=-1)[..., ::2]


        # .cat() -> [[1, 1, 0.01, 0.01, 1, 1, 0.01, 0.01],
        #            [2, 2, 0.02, 0.02, 1, 1, 0.01, 0.01],
        #            [1, 1, 0.01, 0.01, 2, 2, 0.02, 0.02],
        #            [2, 2, 0.02, 0.02, 2, 2, 0.02, 0.02],
        #            [1, 1, 0.01, 0.01, 0, 0,  0.0,  0.0]] -> shape = 5, 1, 8

        # [..., ::2] -> [[[1, 0.01, 1, 0.01]],
        #                [[2, 0.02, 1, 0.01]],
        #                [[1, 0.01, 2, 0.02]],
        #                [[2, 0.02, 2, 0.02]],
        #                [[1, 0.01, 0,  0.0]]] -> freqs, shape = 5, 1, 4




        freqs = freqs.masked_fill(image_index.reshape(-1, 1, 1) < 0, 0)

        # .reshape() < 0 -> [[[False]],
        #                    [[False]],
        #                    [[False]],
        #                    [[False]],
        #                    [[True]]] -> shape = 5, 1, 1

        # [[[1, 0.01, 1, 0.01]],
        #  [[2, 0.02, 1, 0.01]],
        #  [[1, 0.01, 2, 0.02]],
        #  [[2, 0.02, 2, 0.02]],
        #  [[0,    0, 0,    0]]] -> freqs


        # [[[cos(1), cos(0.01), cos(1), cos(0.01)]],
        #  [[cos(2), cos(0.02), cos(1), cos(0.01)]],
        #  [[cos(1), cos(0.01), cos(2), cos(0.02)]],
        #  [[cos(2), cos(0.02), cos(2), cos(0.02)]],
        #  [[cos(0), cos(0.00), cos(0), cos(0.00)]]] -> .cos(freqs), shape = 5, 1, 4


        # [[[sin(1), sin(0.01), sin(1), sin(0.01)]],
        #  [[sin(2), sin(0.02), sin(1), sin(0.01)]],
        #  [[sin(1), sin(0.01), sin(2), sin(0.02)]],
        #  [[sin(2), sin(0.02), sin(2), sin(0.02)]],
        #  [[sin(0), sin(0.00), sin(0), sin(0.00)]]] -> .sin(freqs), shape = 5, 1, 4


        # .stack(dim=-1) -> [


        #                         [[

        #                                 [   cos(1),    sin(1)],
        #                                 [cos(0.01), sin(0.01)],
        #                                 [   cos(1),    sin(1)],
        #                                 [cos(0.01), sin(0.01)]

        #                         ]],

        #                         [[
        #                                 [   cos(2),    sin(2)],
        #                                 [cos(0.02), sin(0.02)],
        #                                 [   cos(1),    sin(1)],
        #                                 [cos(0.01), sin(0.01)]
        #                         ]],

        #                         [[
        #                                 [   cos(1),    sin(1)],
        #                                 [cos(0.01), sin(0.01)],
        #                                 [   cos(2),    sin(2)],
        #                                 [cos(0.02), sin(0.02)]
        #                         ]],

        #                         [[
        #                                 [   cos(2),    sin(2)],
        #                                 [cos(0.02), sin(0.02)],
        #                                 [   cos(2),    sin(2)],
        #                                 [cos(0.02), sin(0.02)]
        #                         ]],

        #                         [[
        #                                 [   cos(0),    sin(0)],
        #                                 [cos(0.00), sin(0.00)],
        #                                 [   cos(0),    sin(0)],
        #                                 [cos(0.00), sin(0.00)]
        #                         ]]

        #                    ]




        self.freq_cis = torch.view_as_complex(torch.stack([torch.cos(freqs), torch.sin(freqs)], dim=-1))


        # [
        #     [[
        #         [cos(1),       j*sin(1)],
        #         [cos(0.01), j*sin(0.01)],
        #         [cos(1),       j*sin(1)],
        #         [cos(0.01), j*sin(0.01)]
        #     ]],

        #     [[
        #         [cos(2),       j*sin(2)],
        #         [cos(0.02), j*sin(0.02)],
        #         [cos(1),       j*sin(1)],
        #         [cos(0.01), j*sin(0.01)]
        #     ]],

        #     [[
        #         [cos(1),       j*sin(1)],
        #         [cos(0.01), j*sin(0.01)],
        #         [cos(2),       j*sin(2)],
        #         [cos(0.02), j*sin(0.02)]
        #     ]],

        #     [[
        #         [cos(2),       j*sin(2)],
        #         [cos(0.02), j*sin(0.02)],
        #         [cos(2),       j*sin(2)],
        #         [cos(0.02), j*sin(0.02)]
        #     ]],

        #     [[
        #         [cos(0),       j*sin(0)],
        #         [cos(0.00), j*sin(0.00)],
        #         [cos(0),       j*sin(0)],
        #         [cos(0.00), j*sin(0.00)]
        #     ]]
        # ] -> freq_cis



    def forward(self, x):
        return self.freq_cis.to(x.device)




In [ ]:
def vision_apply_rotary_emb(query, key, freqs_ci):

    query_ = torch.view_as_complex(query.float().reshape(*query.shape[:-1], -1, 2))
    key_ = torch.view_as_complex(key.float().reshape(*key.shape[:-1], -1, 2))

    freqs_ci = freqs_ci.unsqueeze(0)

    query_out = torch.view_as_real(query_ * freqs_ci).flatten(3)
    key_out = torch.view_as_real(key_ * freqs_ci).flatten(3)

    return query_out.type_as(query), key_out.type_as(key)

In [ ]:
class Llama4VisionAttention(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.config = config
        self.embed_dim = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.num_key_value_groups = 1
        self.attention_dropout = config.attention_dropout

        self.q_proj = nn.Linear(self.embed_dim, self.num_heads * self.head_dim, bias=True)
        self.k_proj = nn.Linear(self.embed_dim, self.num_heads * self.head_dim, bias=True)
        self.v_proj = nn.Linear(self.embed_dim, self.num_heads * self.head_dim, bias=True)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.embed_dim, bias=True)


    def forward(self,
                hidden_states,
                freqs_ci):

        # hidden_states -> (batch_size, seq_len, embed_dim)


        input_shape = hidden_states.shape[:-1]                  # (batch_size, seq_len)
        hidden_shapes = (*input_shape, -1, self.head_dim)       # (batch_size, seq_len, -1, head_dim)


        query_states = self.q_proj(hidden_states).view(hidden_shapes)   # (batch_size, seq_len, num_heads * head_dim) -> (batch_size, seq_len, num_heads, head_dim)
        key_states = self.k_proj(hidden_states).view(hidden_shapes)     # (batch_size, seq_len, num_heads * head_dim) -> (batch_size, seq_len, num_heads, head_dim)
        value_states = self.v_proj(hidden_states).view(hidden_shapes)   # (batch_size, seq_len, num_heads * head_dim) -> (batch_size, seq_len, num_heads, head_dim)


        query_states, key_states = vision_apply_rotary_emb(query_states, key_states, freqs_ci)


        query_states = query_states.transpose(1, 2)     # (batch_size, num_heads, seq_len, head_dim)
        key_states = key_states.transpose(1, 2)         # (batch_size, num_heads, seq_len, head_dim)
        value_states = value_states.transpose(1, 2)     # (batch_size, num_heads, seq_len, head_dim)

        attn_output = F.scaled_dot_product_attention(query_states, key_states, value_states)

        attn_output = attn_output.transpose(1, 2).flatten(2)        # (batch_size, seq_len, num_heads, head_dim) -> (batch_size, seq_len, num_heads * head_dim)

        return attn_output          # (batch_size, seq_len, num_heads * head_dim)



In [ ]:
class Llama4VisionEncoderLayer(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.hidden_size = config.hidden_size
        self.self_attn = Llama4VisionAttention(config)
        self.mlp = Llama4VisionMLP(config)

        self.input_layernorm = nn.LayerNorm(config.hidden_size)
        self.post_attention_layernorm = nn.LayerNorm(config.hidden_size)

    def forward(self, hidden_states, freqs_ci):

        residual = hidden_states

        hidden_states = self.input_layernorm(hidden_states)

        hidden_states = self.self_attn(
            hidden_states, freqs_ci
        )

        hidden_states = hidden_states + residual

        residual = hidden_states

        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = residual + hidden_states

        return hidden_states

In [ ]:
class Llama4VisionEncoder(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.config = config

        self.layers =  nn.ModueList(
            [
                Llama4VisionEncoderLayer(config) for _ in range(config.num_hidden_layers)
            ]
        )

    def forward(self, hidden_states, freqs_ci):

        for layer in self.layers:
            hidden_states = layer(
                hidden_states, freqs_ci
            )

        return hidden_states



In [ ]:
class Llama4UnfoldConvolution(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.unfold = nn.Unfold(kernel_size=config.patch_size, stride=config.patch_size)

        self.linear = nn.Linear(config.num_channels * config.patch_size**2, config.hidden_size, bias=False)


    def forward(self, x):

        unfolded = self.unfold(x).transpose(-1, -2)

        x = self.linear(unfolded)

        return x



In [ ]:
class Llama4VisionModule(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.num_patches = (config.image_size // config.patch_size)**2 + 1

        self.patch_embeddings = Llama4UnfoldConvolution(config)

        self.class_embeddings = nn.Parameter(torch.randn(config.hidden_size))

        self.positional_embedding_vlm = nn.Parameter(torch.randn(self.num_patches, config.hidden_size))

        self.rotary_embed = Llama4VisionRotaryEmbedding(config)

        self.encoder = Llama4VisionEncoder(config)


        # layernorm
        self.layernorm_pre = nn.LayerNorm(config.hidden_size)
        self.layernorm_post = nn.LayerNorm(config.hidden_size)

        # encoders
        self.model = Llama4VisionEncoder(config)
        self.vision_adapter = Llama4VisionPixelShuffleMLP(config)


    def forward(self, pixel_values):

        batch_size, num_channels, height, width = pixel_values.shape

        patch_embed = self.patch_embeddings(pixel_values)

        batch_size, num_patches, embed_dim = patch_embed.shape

        cls_token = self.class_embeddings.expand(batch_size, 1, embed_dim)

        hidden_states = torch.cat([patch_embed, cls_token], dim=1)

        num_patches += 1

        hidden_states = hidden_states + self.positional_embedding_vlm

        hidden_state = self.layernorm_pre(hidden_states)

        freqs = self.rotary_embed(hidden_states)

        output = self.model(hidden_state, freqs)

        output = self.layernorm_post(output)

        output = output[:, :-1]

        adapted = self.vision_adapter(output)

        return adapted


In [ ]:
class Llama4ConditionalGeneration(nn.Module):

    def __init__(self,
                 vision_config,
                 text_config,
                 boi_token_index=200080,
                 eoi_token_index=200081,
                 image_token_index=200092):
        super().__init__()

        self.vision_model = Llama4VisionModel(vision_config)
        self.multi_model_projector = Llama4MultiModalProjection(vision_config, text_config)
        self.language_model = LlamaForCausalLM(text_config)

        self.vocab_size = text_config.vocab_size
        self.pad_token_id = text_config.pad_token_id
        self.boi_token_idx = boi_token_index
        self.eoi_token_idx = eoi_token_index
        self.image_token_idx = image_token_index

    def forward(self,
                input_ids,
                pixel_values=None,
                attention_mask=None,
                position_ids=None,
                past_key_values=None):

        token_embeddings = self.language_model.model.embed_tokens(input_ids)

        if pixel_values is not None:
            image_features = self.vision_model(pixel_values)

            text_embed_shape = token_embeddings.shape

            image_flat = image_features.reshape(-1, image_features.shape[-1])

            proj_image_flat = self.multi_modal_projector(image_flat)

            special_image_mask = (input_ids == self.image_token_index).unsqueeze(-1)

            final_mask = special_image_mask.to(token_embeddings.device)

            token_embeddings = token_embeddings(-1, token_embeddings.size(-1))

            final_mask_id = final_mask.squeeze(-1).reshape(-1)

            num_tokens_to_fill = final_mask.sum()

            assert num_tokens_to_fill == proj_image_flat.shape[0]

            expanded_mask = final_mask_id.unsqueeze(-1).expand(-1, token_embeddings.shape[-1])

            token_embeddings.mask_scatter_(expanded_mask, proj_image_flat)

            token_embeddings = token_embeddings.reshape(text_embed_shape)

        outputs, past_key_values = self.language_model(
            input_embeds=token_embeddings,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values
        )

        return outputs, past_key_values